# Example: feature-source substitution impact assessment

This notebook runs the framework end to end on synthetic in-memory data. It defines a model, migration, and experiment; produces old/new scores and GH bands; executes all configured analyses; and writes an Excel report.

The synthetic features deliberately contain several negative sentinel values. Each distinct value is kept as its own migration-matrix category, independently on the old and new axes.

In [1]:
from pathlib import Path
import sys

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src' / 'troca_fontes').exists():
    raise RuntimeError('Run this notebook with the project root as the working directory.')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from troca_fontes import analyse_reference
from troca_fontes.config import (
    ExperimentConfig, MigrationConfig, ModelConfig, NewSourceSpec,
    PopulationSpec, ReferenceSpec, SegmentationSpec, TableRef,
)
from troca_fontes.preparation import NEW_GH_COL, NEW_SCORE_COL
from troca_fontes.report import workbook as wb

OUTPUT_DIR = PROJECT_ROOT / 'example_output'
OUTPUT_DIR.mkdir(exist_ok=True)
print(f'Project root: {PROJECT_ROOT}')

Project root: C:\Users\tms19\OneDrive\Documents\ChatGPT\pics code extraction\recovered_project


## 1. Create synthetic old/new feature values and train a model

The production framework normally loads a trained model and extracted reference tables. Here we generate equivalent inputs locally so the example is reproducible.

In [2]:
rng = np.random.default_rng(20260826)
n = 600

income_old = rng.lognormal(mean=8.1, sigma=0.55, size=n)
util_old = rng.beta(2.0, 5.0, size=n)
age = rng.integers(18, 76, size=n).astype(float)

# Distinct negative values represent distinct source-system sentinels.
income_sentinel = rng.random(n) < 0.12
income_old[income_sentinel] = rng.choice([-999.0, -2.0, -1.0], income_sentinel.sum())
util_sentinel = rng.random(n) < 0.10
util_old[util_sentinel] = rng.choice([-9.0, -1.0], util_sentinel.sum())

income_new = income_old.copy()
income_nonnegative = income_new >= 0
income_new[income_nonnegative] *= rng.normal(1.03, 0.08, income_nonnegative.sum())
income_switch = (income_new < 0) & (rng.random(n) < 0.35)
income_new[income_switch] = rng.choice([-999.0, -2.0, -1.0], income_switch.sum())

util_new = util_old.copy()
util_nonnegative = util_new >= 0
util_new[util_nonnegative] = np.clip(
    util_new[util_nonnegative] + rng.normal(0.015, 0.04, util_nonnegative.sum()), 0, 1
)
util_switch = (util_new < 0) & (rng.random(n) < 0.40)
util_new[util_switch] = rng.choice([-9.0, -3.0, -1.0], util_switch.sum())

channel = rng.choice(['APP', 'BRANCH', 'PARTNER', None], n, p=[0.48, 0.30, 0.17, 0.05])
is_customer = (rng.random(n) < 0.72).astype(int)

income_signal = np.zeros(n, dtype=float)
valid_income = income_old >= 0
income_signal[valid_income] = np.log1p(income_old[valid_income])
util_signal = np.where(util_old >= 0, util_old, 0.5)
logit = -2.5 + 2.7 * util_signal - 0.12 * income_signal + 0.012 * (age - 40)
bad_probability = 1 / (1 + np.exp(-logit))
target_full = rng.binomial(1, bad_probability)

feature_order = ['income_ft', 'utilization_ft', 'age']
training_frame = pd.DataFrame({
    'income_ft': income_old, 'utilization_ft': util_old, 'age': age
})
classifier = RandomForestClassifier(
    n_estimators=100, min_samples_leaf=8, random_state=20260826
)
classifier.fit(training_frame, target_full)

model_path = OUTPUT_DIR / 'synthetic_model.joblib'
joblib.dump(classifier, model_path)
print(f'Synthetic model saved to: {model_path}')

Synthetic model saved to: C:\Users\tms19\OneDrive\Documents\ChatGPT\pics code extraction\recovered_project\example_output\synthetic_model.joblib


## 2. Define the framework model, migration, and experiment

`values` is intentionally omitted from the segmentation specification, so `APP`, `BRANCH`, and `PARTNER` are discovered from the data automatically.

In [3]:
old_score = classifier.predict_proba(training_frame)[:, 1]
quantile_points = np.quantile(old_score, np.linspace(0, 1, 7))
gh_cutpoints = tuple(np.unique(np.concatenate(([0.0], quantile_points, [1.0]))))

model = ModelConfig(
    name='synthetic_credit_model',
    display_name='SYNTHETIC CREDIT MODEL',
    pkl_path=str(model_path),
    flat_table=TableRef(engine='teradata', name='example.synthetic_flat_table'),
    id_cols=('customer_id', 'reference_date'),
    score_col='score',
    gh_col='gh',
    cutpoints_gh=gh_cutpoints,
    populations=(
        PopulationSpec(key='all', display_name='ALL CUSTOMERS'),
        PopulationSpec(key='customers', display_name='ACTIVE CUSTOMERS', flag_col='is_customer'),
        PopulationSpec(key='labeled', display_name='LABELED SAMPLE', target_col='target'),
    ),
)

migration = MigrationConfig(
    name='synthetic_book',
    display_name='SYNTHETIC NEW BOOK',
    new_source=NewSourceSpec(engine='athena', table='example.synthetic_new_book'),
    feature_names=('income_ft', 'utilization_ft'),
    segmentation=SegmentationSpec(
        column='CHANNEL', display_name='CHANNEL', include_null=True
    ),
)

reference = ReferenceSpec(
    key='example', label='SYNTHETIC REFERENCE', date='2026-08-26',
    anomesdia=20260826, include_target=True,
)
experiment = ExperimentConfig(
    name='synthetic_example',
    model=model.name,
    migrations=(migration.name,),
    report_sheet='SYNTHETIC EXAMPLE',
    references=(reference,),
    populations=('all', 'customers', 'labeled'),
    analyses=(
        'score_gh_match', 'gh_migration_matrix', 'performance_gini',
        'per_variable_drift', 'feature_migration_matrix',
    ),
)

display(pd.DataFrame({
    'object': ['model', 'migration', 'experiment'],
    'name': [model.name, migration.name, experiment.name],
}))

,object,name
0,model,synthetic_credit_model
1,migration,synthetic_book
2,experiment,synthetic_example


## 3. Assemble the evaluation frame and calculate old/new scores and GH

The unsuffixed feature columns are old-source values. The `_ste` columns are new-source values, matching the framework's preparation convention.

In [4]:
def assign_gh(scores, cutpoints):
    return np.searchsorted(np.asarray(cutpoints)[1:-1], scores, side='right') + 1

new_scoring_frame = training_frame.copy()
new_scoring_frame['income_ft'] = income_new
new_scoring_frame['utilization_ft'] = util_new
new_score = classifier.predict_proba(new_scoring_frame)[:, 1]

target = target_full.astype(float)
target[rng.random(n) < 0.15] = np.nan
eval_df = pd.DataFrame({
    'customer_id': np.arange(1, n + 1),
    'reference_date': pd.Timestamp('2026-08-26'),
    'income_ft': income_old,
    'income_ft_ste': income_new,
    'utilization_ft': util_old,
    'utilization_ft_ste': util_new,
    'age': age,
    'channel': channel,
    'is_customer': is_customer,
    'target': target,
    'score': old_score,
    'gh': assign_gh(old_score, gh_cutpoints),
    NEW_SCORE_COL: new_score,
    NEW_GH_COL: assign_gh(new_score, gh_cutpoints),
})

display(eval_df.head())
print(f'Evaluation rows: {len(eval_df):,}')

,customer_id,reference_date,income_ft,income_ft_ste,utilization_ft,utilization_ft_ste,age,channel,is_customer,target,score,gh,score_mdl_new_src,GH_new_src
0,1,2026-08-26,4375.767864,3826.938501,0.165774,0.158669,57.0,APP,1,NaN,0.027525,3,0.029499,3
1,2,2026-08-26,3142.828547,3657.324084,0.098156,0.100857,23.0,PARTNER,1,0.0,0.042409,4,0.037770,3
2,3,2026-08-26,1284.381932,1322.082074,0.114871,0.194841,41.0,APP,0,0.0,0.099002,6,0.069394,5
3,4,2026-08-26,1884.085380,1941.964618,0.408358,0.504838,60.0,PARTNER,0,0.0,0.049043,4,0.130045,6
4,5,2026-08-26,4810.056675,4952.367720,0.280796,0.257372,60.0,BRANCH,1,0.0,0.078506,5,0.102013,6


Evaluation rows: 600


## 4. Run the configured analysis

In [5]:
importance_map = dict(zip(feature_order, classifier.feature_importances_))
importance_total = sum(importance_map.values())
importance_map = {key: value / importance_total for key, value in importance_map.items()}

blocks = analyse_reference(
    eval_df,
    model=model,
    experiment=experiment,
    include_target=reference.include_target,
    features=list(migration.feature_names),
    importances=importance_map,
    segmentation=migration.segmentation,
)

print('Discovered analysis segments:', blocks['segments'])
display(pd.DataFrame(blocks['match']['all']).T)

Discovered analysis segments: ('TOTAL', 'APP', 'BRANCH', 'PARTNER', 'NULL')


,volume,scores_6a,scores_1pp,gh_menos1,gh_iguais,gh_mais1
TOTAL,600.0,0.066667,0.516667,0.138333,0.598333,0.165000
APP,294.0,0.071429,0.500000,0.108844,0.622449,0.166667
BRANCH,191.0,0.062827,0.497382,0.157068,0.586387,0.172775
PARTNER,90.0,0.044444,0.600000,0.166667,0.544444,0.155556
NULL,25.0,0.120000,0.560000,0.240000,0.600000,0.120000


### Inspect one feature migration matrix

Notice that each negative value has its own row or column category. The old and new sentinel sets are allowed to differ.

In [6]:
income_matrix = blocks['feature_migration']['all']['TOTAL']['income_ft']
display(income_matrix.matrix)
print('Old nonnegative decile edges:', income_matrix.old_edges)
print('New nonnegative decile edges:', income_matrix.new_edges)
print('Matrix population total:', int(income_matrix.matrix.to_numpy().sum()))

income_ft_new,SENTINEL (-999),SENTINEL (-2),SENTINEL (-1),D01,D02,D03,D04,D05,D06,D07,D08,D09,D10,MISSING
income_ft_old,,,,,,,,,,,,,,
SENTINEL (-999),19,4,5,0,0,0,0,0,0,0,0,0,0,0
SENTINEL (-2),1,22,3,0,0,0,0,0,0,0,0,0,0,0
SENTINEL (-1),0,1,18,0,0,0,0,0,0,0,0,0,0,0
D01,0,0,0,49,4,0,0,0,0,0,0,0,0,0
D02,0,0,0,4,41,8,0,0,0,0,0,0,0,0
D03,0,0,0,0,8,29,15,0,0,0,0,0,0,0
D04,0,0,0,0,0,15,28,10,0,0,0,0,0,0
D05,0,0,0,0,0,0,10,30,12,1,0,0,0,0
D06,0,0,0,0,0,0,0,13,29,10,0,0,0,0


Old nonnegative decile edges: (712.9096340789323, 1773.4253564508256, 2138.7327067270594, 2545.836342446155, 3029.9714551606717, 3433.102167812134, 3862.8258042160037, 4433.809411858907, 5232.732105915229, 6720.126210562872, 18081.484691112444)
New nonnegative decile edges: (718.7428150684113, 1728.6522413423245, 2199.631201856635, 2577.910793533082, 3095.517336870057, 3517.16798800748, 4006.4179562642707, 4580.3015567336815, 5464.390444822119, 6794.950812435275, 18511.464898211154)
Matrix population total: 600


## 5. Create the Excel report

In [7]:
reference_report = wb.ReferenceReport(
    banner='ANÁLISES - SYNTHETIC REFERENCE',
    match=blocks['match'],
    performance=blocks['performance'],
    drift=blocks['drift'],
    gh_migration=blocks['gh_migration'],
    feature_migration=blocks['feature_migration'],
    segments=blocks['segments'],
)

counts = {
    (population, reference.key, segment): (
        int(metrics['volume']) if pd.notna(metrics['volume']) else 0
    )
    for population, by_segment in blocks['match'].items()
    for segment, metrics in by_segment.items()
}
sheet_report = wb.SheetReport(
    sheet_name=experiment.report_sheet,
    model_display=model.display_name,
    book_display=migration.display_name,
    publicos=[f'- {model.population(key).display_name}' for key in experiment.populations],
    referencias=[reference.label],
    ref_labels=[reference.key.upper()],
    volumetrias={
        'groups': [(key, model.population(key).display_name) for key in experiment.populations],
        'ref_keys': [reference.key],
        'counts': counts,
        'segments': list(blocks['segments']),
        'segmentation_name': migration.segmentation.display_name,
    },
    notes=[
        '- Synthetic data only; no production systems were queried.',
        '- Each distinct negative sentinel is reported as a separate category.',
    ],
    features=list(migration.feature_names),
    references=[reference_report],
)

report_path = OUTPUT_DIR / 'example_migration_report.xlsx'
workbook = wb.build_workbook([sheet_report])
workbook.save(report_path)
print(f'Report created: {report_path}')
print('Workbook sheets:', workbook.sheetnames)

Report created: C:\Users\tms19\OneDrive\Documents\ChatGPT\pics code extraction\recovered_project\example_output\example_migration_report.xlsx
Workbook sheets: ['SYNTHETIC EXAMPLE', 'SYNTHETIC EXAMPLE - MIG GH', 'SYNTHETIC EXAMPLE - MIG VAR']
